In [1]:
%load_ext autoreload
%autoreload 2


import os
import logging
import pandas as pd
import numpy as np
import time
import torch
import wandb
import copy
from tqdm import tqdm


from src.constants import DEVICE
from src.utils.loss_func.get_loss_function import get_loss_function
from src.evaluation.mainMetricHandler import mainMetricHandler
from src.dataset.load_dataset import load_dataset, generate_K_fold_cross_validation_splits
from src.models.tools.get_classification_model import get_classification_model
from src.dataset.get_dataloader import make_dataloader   
from src.dataset.get_transforms import get_transforms
from src.utils.loss_func.get_loss_function import get_loss_function
from src.evaluation.mainMetricHandler import mainMetricHandler



/home/macraedc/.predRTenv/lib64/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import torch

torch.cuda.is_available()


True

In [3]:
from src.config_presets.tools.get_config import get_config

config = get_config('Trial32_Config_VM')




src/config_presets/Base_config.yaml
src/config_presets/Trial32_Config_VM.yaml


In [4]:
# load the data
df_train_val, df_test = load_dataset(config)
    
dataset_split_dict = generate_K_fold_cross_validation_splits(config, df_train_val)

# cap the number of iterations, if it is less than the number of k-splits to make
n_iterations = config['data']['kFolds']['n_iterations']

# get the data transforms
train_transforms, val_transforms = get_transforms(config)
# get the loss function
loss_function = get_loss_function(config)

metricHandler = mainMetricHandler(config)



Removed patients (no image data) = 0
Train/Val dataset 872 (80.0%), Test dataset 218 (20.0%)


/home/macraedc/.predRTenv/lib64/python3.9/site-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


In [5]:
train_data, val_data = dataset_split_dict[0]['train'], dataset_split_dict[0]['val']

train_loader, metadata = make_dataloader(config, train_data, train_transforms, validation_mode=False)
val_loader, _ = make_dataloader(config, val_data, val_transforms, validation_mode=True)



In [1]:
import torch

torch.cuda.empty_cache()

In [7]:
%load_ext autoreload
%autoreload 2


from src.models.tools.get_classification_model import get_classification_model

config['model']['model_name'] = 'TransRP'

config['general']['resultsCurrentDirectory'] = os.path.join(os.getcwd(), 'VM_results')


config['model']['TransRP']['clinical_features_method'] = 'cls'

config['model']['TransRP']['vit_dim'] = 128
config['model']['TransRP']['cls_gating'] = False

config["model"]["TransRP"]["cls_merge_image_features"] = False
config["model"]["TransRP"]["cls_per_class_weighting"] = True



config['model']['linear_units'] = []

config['training']['batch_size'] = 2

model = get_classification_model(config, metadata=metadata, save_summary=True)



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
tensor([ 2.6297,  0.7559,  2.3541, -0.0891, -1.2963,  0.1920, -0.4607, -0.1370,
        -1.2131,  0.8546,  0.6519, -1.0948,  1.2689, -1.7148, -0.4087,  0.9270,
         1.5527, -0.0905, -1.1477,  0.6510, -1.1400, -2.2770,  0.4873, -0.6062,
         0.0963, -0.6254, -2.6437, -0.2096, -1.3815, -0.5687, -0.3291, -1.0697,
         0.1359,  0.4432,  1.2167, -0.1493,  0.3237,  0.3281,  0.7859, -0.6466,
         0.3697,  0.5549, -0.5439,  1.1685, -0.4063,  0.0989, -1.5316,  0.6023,
         0.3646,  0.2386,  0.5751,  0.5344, -0.4438,  0.2606, -0.0885, -0.0288,
         0.9910,  0.2459,  1.4059, -0.4720, -0.1321, -0.5201,  0.7715,  1.3881,
        -1.7617, -0.6615, -0.9502, -0.3945,  1.5370,  0.1257, -0.5660,  0.5481,
        -1.7142, -1.6133,  0.6479, -2.0596, -0.9019,  0.7374, -0.0773,  0.8303,
         0.3835, -0.4385, -0.7286,  0.4961,  1.3843,  0.7508,  1.3949,  2.4538,
        -1.0464, -0.1332, -0.840

In [94]:

for name, module in model.encoder.named_modules():
    last_module = module
print(last_module)

AdaptiveAvgPool3d(output_size=(1, 1, 1))


In [78]:
model.output_head.linear_layers.shared_fc_layers.[4]

SyntaxError: invalid syntax (2619673146.py, line 1)

In [79]:
getattr(model.output_head.linear_layers.shared_fc_layers, torch.nn.Linear)

TypeError: getattr(): attribute name must be string

In [76]:
model.output_head.linear_layers.shared_fc_layers
last_linear_layer = None
last_name = None
for name, layer in model.output_head.linear_layers.shared_fc_layers.named_modules():
    if isinstance(layer, (torch.nn.Linear, torch.nn.LazyLinear)):
        last_linear_layer = layer
        last_name = name

print(last_name)
print(last_linear_layer)

Linear_shared_2
Linear(in_features=128, out_features=64, bias=True)


In [81]:
model.output_head.linear_layers.shared_fc_layers.name

AttributeError: 'ModuleList' object has no attribute 'name'

In [ ]:
getattr(model.output_head.transformer.layers, 'Attention Block 3')[1]

In [ ]:
model.output_head.linear_layers.shared_fc_layers


In [66]:
def find_last_module(module, target_name='endpoint_heads'):
    last_module = None
    for name, submodule in module.named_children():
        if target_name in name:
            break
        last_module = submodule
        
        # Recursively search in submodules
        found_module = find_last_module(submodule, target_name)
        if found_module:
            last_module = found_module
            print(name)
    return last_module

last_module = find_last_module(model)
print(last_module)

0
block0
downsample
0
block1
downsample
0
block2
downsample
0
block3
backbone
encoder
MultiToxOutputHead(
  (clinical_variables_shared_fc_layers): ModuleList()
  (shared_fc_layers): ModuleList()
  (endpoint_heads): ModuleDict(
    (Aspiration_M06): ModuleList(
      (0): Dropout(p=0.1, inplace=False)
      (1): Linear(in_features=273, out_features=64, bias=True)
      (2): LeakyReLU(negative_slope=0.1)
      (3): Dropout(p=0.1, inplace=False)
      (4): Linear(in_features=64, out_features=1, bias=True)
    )
    (Dysphagia_M06): ModuleList(
      (0): Dropout(p=0.1, inplace=False)
      (1): Linear(in_features=273, out_features=64, bias=True)
      (2): LeakyReLU(negative_slope=0.1)
      (3): Dropout(p=0.1, inplace=False)
      (4): Linear(in_features=64, out_features=1, bias=True)
    )
    (Sticky_M06): ModuleList(
      (0): Dropout(p=0.1, inplace=False)
      (1): Linear(in_features=273, out_features=64, bias=True)
      (2): LeakyReLU(negative_slope=0.1)
      (3): Dropout(p=0.1,

In [14]:
for name, module in model.output_head.named_children():
    print(name)

patch_embedding
transformer
norm
linear_layers
clc_embed


In [71]:
model.output_head.shared_fc_layers

ModuleList()

In [29]:


for name, module in model.output_head.transformer.layers.named_modules():
    print(name)


Attention Block 0
Attention Block 0.0
Attention Block 0.0.norm
Attention Block 0.0.attend
Attention Block 0.0.dropout
Attention Block 0.0.to_qkv
Attention Block 0.0.to_out
Attention Block 0.0.to_out.0
Attention Block 0.0.to_out.1
Attention Block 0.1
Attention Block 0.1.net
Attention Block 0.1.net.0
Attention Block 0.1.net.1
Attention Block 0.1.net.2
Attention Block 0.1.net.3
Attention Block 0.1.net.4
Attention Block 0.1.net.5
Attention Block 1
Attention Block 1.0
Attention Block 1.0.norm
Attention Block 1.0.attend
Attention Block 1.0.dropout
Attention Block 1.0.to_qkv
Attention Block 1.0.to_out
Attention Block 1.0.to_out.0
Attention Block 1.0.to_out.1
Attention Block 1.1
Attention Block 1.1.net
Attention Block 1.1.net.0
Attention Block 1.1.net.1
Attention Block 1.1.net.2
Attention Block 1.1.net.3
Attention Block 1.1.net.4
Attention Block 1.1.net.5
Attention Block 2
Attention Block 2.0
Attention Block 2.0.norm
Attention Block 2.0.attend
Attention Block 2.0.dropout
Attention Block 2.0.t

In [68]:
att_layer_name = f"Attention Block {config['model']['TransRP']['vit_depth'] - 1}"

In [33]:
for name, layer in model.named_modules():
    #if isinstance(layer, nn.ReLU):
    print(name, layer)

    pytorch_layer_obj = getattr(model, name)

 MultiTox_Classifier(
  (encoder): DenseNet(
    (features): Sequential(
      (conv0): Conv3d(3, 64, kernel_size=(5, 5, 5), stride=(2, 2, 2), padding=(2, 2, 2), bias=False)
      (norm0): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu0): LeakyReLU(negative_slope=0)
      (pool0): MaxPool3d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (denseblock_1): _DenseBlock(
        (denselayer1): _DenseLayer(
          (layers): Sequential(
            (norm1): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (relu1): LeakyReLU(negative_slope=0)
            (conv1): Conv3d(64, 128, kernel_size=(1, 1, 1), stride=(1, 1, 1), bias=False)
            (norm2): BatchNorm3d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (relu2): LeakyReLU(negative_slope=0)
            (conv2): Conv3d(128, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1), bias=Fa

AttributeError: 'MultiTox_Classifier' object has no attribute ''

In [39]:
model.output_head.transformer.layers[]

TypeError: 'str' object cannot be interpreted as an integer

In [42]:
getattr(model.output_head.transformer.layers, 'Attention Block 1')

ModuleList(
  (0): Attention(
    (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    (attend): Softmax(dim=-1)
    (dropout): Dropout(p=0.2, inplace=False)
    (to_qkv): Linear(in_features=128, out_features=384, bias=False)
    (to_out): Sequential(
      (0): Linear(in_features=128, out_features=128, bias=True)
      (1): Dropout(p=0.2, inplace=False)
    )
  )
  (1): FeedForward(
    (net): Sequential(
      (0): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (1): Linear(in_features=128, out_features=512, bias=True)
      (2): GELU(approximate='none')
      (3): Dropout(p=0.2, inplace=False)
      (4): Linear(in_features=512, out_features=128, bias=True)
      (5): Dropout(p=0.2, inplace=False)
    )
  )
)

In [46]:
attention_block = getattr(model.output_head.transformer.layers, 'Attention Block 3')
print(attention_block)

attention_block[1]

ModuleList(
  (0): Attention(
    (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    (attend): Softmax(dim=-1)
    (dropout): Dropout(p=0.2, inplace=False)
    (to_qkv): Linear(in_features=128, out_features=384, bias=False)
    (to_out): Sequential(
      (0): Linear(in_features=128, out_features=128, bias=True)
      (1): Dropout(p=0.2, inplace=False)
    )
  )
  (1): FeedForward(
    (net): Sequential(
      (0): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (1): Linear(in_features=128, out_features=512, bias=True)
      (2): GELU(approximate='none')
      (3): Dropout(p=0.2, inplace=False)
      (4): Linear(in_features=512, out_features=128, bias=True)
      (5): Dropout(p=0.2, inplace=False)
    )
  )
)


FeedForward(
  (net): Sequential(
    (0): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    (1): Linear(in_features=128, out_features=512, bias=True)
    (2): GELU(approximate='none')
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=512, out_features=128, bias=True)
    (5): Dropout(p=0.2, inplace=False)
  )
)

In [49]:
model.output_head.linear_layers[-1]

TypeError: 'MultiToxOutputHead' object is not subscriptable

In [36]:
model.output_head.transformer.layers.[0].feed_forward.parameters()

SyntaxError: invalid syntax (3408966411.py, line 1)